# 02 · Almost Certain? Models, validation and results
Question: among Kalshi YES contracts first seen at a 90–99¢ ask, do market features (volume, time to the event,
recent price movement, category) predict which favorites lose better than the quoted price itself?

Design: fit on **train** (before Oct 15), choose hyperparameters on **validation** (Oct 15–19) by Brier score,
refit the chosen setting on train + validation, then score the untouched **test** period (Oct 20 onward) once.

In [1]:
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt  # styled via style_guide/ucla_style.py
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score, average_precision_score
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RES = ROOT/'results/midterm'; FIG = RES/'figures'; FIG.mkdir(parents=True, exist_ok=True)
SEED = 414; rng = np.random.default_rng(SEED)
d = pd.read_parquet(ROOT/'data/processed/midterm_entries.parquet')
print(d.groupby('split').agg(markets=('y_win','size'), events=('event_slug','nunique'), losses=('y_win', lambda s: int((1-s).sum()))))

            markets  events  losses
split                              
test           2299    1063     114
train          4358    2131     179
validation     3694    1724     220


## Features
Everything is measured at or before the entry candle. Time-to-event uses the *scheduled* event time; the actual close
time is never a feature. Missing trailing history is filled with 0 plus an explicit indicator.

In [2]:
h = d.hours_to_event.clip(-72, 336)
d['slog_hours_to_event'] = np.sign(h) * np.log1p(h.abs())
d['log_hours_since_open'] = np.log1p(d.hours_since_open.clip(lower=0))
d['has_prior_quote'] = d.prev_mid.notna().astype(int)
d['mid_change_6h_f'] = d.mid_change_6h.fillna(0)
d['fresh_cross_f'] = d.fresh_cross.fillna(0)
PRICE = ['midpoint']
MARKET = ['midpoint','spread','log_open_interest','slog_hours_to_event','log_hours_since_open','has_prior_quote',
          'mid_change_6h_f','fresh_cross_f','prior_candles_6h','log_event_size']
VOLUME = ['log_volume','volume_missing','log_volume_prev_6h']
CAT = ['category_group']
FEATS = {'price_only': (PRICE, []), 'no_volume': (MARKET, CAT), 'with_volume': (MARKET + VOLUME, CAT)}
tr, va, te = (d[d.split == s] for s in ['train','validation','test'])
trva = pd.concat([tr, va])

def prep(num, cat):
    t = [('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num)]
    if cat: t.append(('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=10), cat))
    return ColumnTransformer(t)

def metrics(y, p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return dict(brier=brier_score_loss(y, p), log_loss=log_loss(y, p, labels=[0, 1]),
                auc=roc_auc_score(y, p), loss_ap=average_precision_score(1 - y, 1 - p))

## Model families and validation grids
* **Elastic-net logistic regression**: penalty strength C and L1 share (l1_ratio 0 = ridge, 1 = lasso), Week 3.
* **Random forest** and **XGBoost** (gradient-boosted trees): small depth / leaf / learning-rate grids, Week 4.
Each family is tuned on validation Brier only. The test set is not touched until the end.

In [3]:
def logit(C, l1): return LogisticRegression(penalty='elasticnet', solver='saga', C=C, l1_ratio=l1, max_iter=5000, random_state=SEED)
GRIDS = {
  'elastic_net_logistic': [dict(C=C, l1=l1) for C in [0.01, 0.1, 1, 10] for l1 in [0, 0.5, 1]],
  'random_forest': [dict(max_depth=md, min_samples_leaf=ml) for md in [4, 8] for ml in [20, 50]],
  'xgboost': [dict(max_depth=md, learning_rate=lr, n_estimators=ne) for md in [2, 3] for lr in [0.03, 0.1] for ne in [200]],
}
def make(family, hp):
    if family == 'elastic_net_logistic': return logit(**hp)
    if family == 'random_forest': return RandomForestClassifier(n_estimators=300, n_jobs=4, random_state=SEED, **hp)
    return XGBClassifier(subsample=0.8, colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0, n_jobs=4,
                         random_state=SEED, eval_metric='logloss', **hp)

search, chosen = [], {}
for family, grid in GRIDS.items():
    for fs in ['no_volume', 'with_volume']:
        num, cat = FEATS[fs]
        for hp in grid:
            m = Pipeline([('prep', prep(num, cat)), ('model', make(family, hp))]).fit(tr, tr.y_win)
            search.append(dict(family=family, features=fs, **{k: str(v) for k, v in hp.items()},
                               val_brier=brier_score_loss(va.y_win, m.predict_proba(va)[:, 1])))
search = pd.DataFrame(search)
for (family, fs), g in search.groupby(['family', 'features']):
    best = g.sort_values('val_brier').iloc[0]
    chosen[(family, fs)] = {k: (float(best[k]) if k in ('C', 'l1', 'learning_rate') else int(best[k]))
                            for k in GRIDS[family][0]}
search.to_csv(RES/'validation_search.csv', index=False)
search.sort_values('val_brier').groupby(['family','features']).head(1)

,family,features,C,l1,val_brier,max_depth,min_samples_leaf,learning_rate,n_estimators
13,elastic_net_logistic,with_volume,0.01,0.5,0.055778,NaN,NaN,NaN,NaN
1,elastic_net_logistic,no_volume,0.01,0.5,0.055786,NaN,NaN,NaN,NaN
29,random_forest,with_volume,NaN,NaN,0.055926,4,50,NaN,NaN
24,random_forest,no_volume,NaN,NaN,0.056015,4,20,NaN,NaN
32,xgboost,no_volume,NaN,NaN,0.056063,2,NaN,0.03,200
36,xgboost,with_volume,NaN,NaN,0.056084,2,NaN,0.03,200


## Refit on train + validation, score the test period once

In [4]:
preds = {'midpoint': te.midpoint.values, 'training_win_rate': np.full(len(te), trva.y_win.mean())}
val_preds = {'midpoint': va.midpoint.values}
m = Pipeline([('prep', prep(PRICE, [])), ('model', LogisticRegression(C=1e6, max_iter=5000))])
val_preds['price_only_logistic'] = m.fit(tr, tr.y_win).predict_proba(va)[:, 1]
preds['price_only_logistic'] = m.fit(trva, trva.y_win).predict_proba(te)[:, 1]
fitted = {}
for (family, fs), hp in chosen.items():
    num, cat = FEATS[fs]; name = f'{family}__{fs}'
    m = Pipeline([('prep', prep(num, cat)), ('model', make(family, hp))])
    val_preds[name] = m.fit(tr, tr.y_win).predict_proba(va)[:, 1]
    m = Pipeline([('prep', prep(num, cat)), ('model', make(family, hp))]).fit(trva, trva.y_win)
    preds[name] = m.predict_proba(te)[:, 1]; fitted[name] = m
rows = [dict(split='validation', model=k, **metrics(va.y_win.values, v)) for k, v in val_preds.items()]
rows += [dict(split='test', model=k, **metrics(te.y_win.values, v)) for k, v in preds.items()]
scores = pd.DataFrame(rows); scores.to_csv(RES/'model_scores.csv', index=False)
scores[scores.split == 'test'].sort_values('brier').round(5)

,split,model,brier,log_loss,auc,loss_ap
8,test,midpoint,0.04624,0.18727,0.68490,0.08231
11,test,elastic_net_logistic__no_volume,0.04629,0.18820,0.69040,0.08832
12,test,elastic_net_logistic__with_volume,0.04629,0.18820,0.69039,0.08831
10,test,price_only_logistic,0.04631,0.18739,0.68490,0.08231
14,test,random_forest__with_volume,0.04658,0.18924,0.66903,0.07317
13,test,random_forest__no_volume,0.04662,0.18949,0.65975,0.07078
16,test,xgboost__with_volume,0.04677,0.18685,0.69137,0.07754
15,test,xgboost__no_volume,0.04687,0.18708,0.69242,0.07677
9,test,training_win_rate,0.04713,0.19730,0.50000,0.04959


## Uncertainty: event-level bootstrap of Brier differences on test
Markets in one event share an outcome driver, so whole events are resampled (1,000 replicates).

In [5]:
ev = te.event_slug.values; uev = np.unique(ev); idx = {e: np.where(ev == e)[0] for e in uev}; y = te.y_win.values
def boot_diff(pa, pb, B=1000):
    out = []
    for _ in range(B):
        ii = np.concatenate([idx[e] for e in rng.choice(uev, len(uev))])
        out.append(np.mean((pa[ii] - y[ii])**2) - np.mean((pb[ii] - y[ii])**2))
    out = np.array(out); return float(np.mean((pa - y)**2) - np.mean((pb - y)**2)), *np.percentile(out, [2.5, 97.5]).tolist()
cmp = []
for k in preds:
    if k != 'midpoint':
        pt, lo, hi = boot_diff(preds[k], preds['midpoint']); cmp.append(dict(model=k, brier_minus_midpoint=pt, ci_low=lo, ci_high=hi))
for fam in GRIDS:
    pt, lo, hi = boot_diff(preds[f'{fam}__with_volume'], preds[f'{fam}__no_volume'])
    cmp.append(dict(model=f'{fam}: with minus without volume', brier_minus_midpoint=pt, ci_low=lo, ci_high=hi))
cmp = pd.DataFrame(cmp); cmp.to_csv(RES/'bootstrap_comparisons.csv', index=False); cmp.round(5)

,model,brier_minus_midpoint,ci_low,ci_high
0,training_win_rate,0.00088,0.00027,0.00150
1,price_only_logistic,0.00006,-0.00013,0.00028
2,elastic_net_logistic__no_volume,0.00005,-0.00015,0.00025
3,elastic_net_logistic__with_volume,0.00005,-0.00013,0.00027
4,random_forest__no_volume,0.00038,-0.00035,0.00103
5,random_forest__with_volume,0.00034,-0.00027,0.00092
6,xgboost__no_volume,0.00063,-0.00010,0.00148
7,xgboost__with_volume,0.00053,-0.00020,0.00125
8,elastic_net_logistic: with minus without volume,0.00000,-0.00000,0.00000
9,random_forest: with minus without volume,-0.00004,-0.00018,0.00008


## What the best model learned
Standardized elastic-net coefficients (positive = more likely to win) and XGBoost permutation importance on test.

In [6]:
en = fitted['elastic_net_logistic__with_volume']
names = en.named_steps['prep'].get_feature_names_out(); coef = en.named_steps['model'].coef_[0]
coefs = pd.Series(coef, index=[n.split('__', 1)[1] for n in names]).sort_values()
coefs.to_csv(RES/'elastic_net_coefficients.csv'); print(chosen[('elastic_net_logistic','with_volume')]); coefs.round(3)

{'C': 0.01, 'l1': 0.5}


log_event_size                       -0.049
spread                               -0.031
volume_missing                        0.000
category_group_Other                  0.000
category_group_Financials             0.000
category_group_Entertainment          0.000
category_group_Elections              0.000
category_group_Economics              0.000
category_group_Crypto                 0.000
category_group_Commodities            0.000
category_group_Climate and Weather    0.000
log_volume_prev_6h                    0.000
category_group_Sports                 0.000
log_volume                            0.000
prior_candles_6h                      0.000
fresh_cross_f                         0.000
mid_change_6h_f                       0.000
has_prior_quote                       0.000
log_hours_since_open                  0.000
slog_hours_to_event                   0.000
category_group_Politics               0.000
log_open_interest                     0.010
midpoint                        

In [7]:
from sklearn.inspection import permutation_importance
xg = fitted['xgboost__with_volume']; cols = MARKET + VOLUME + CAT
pi = permutation_importance(xg, te[cols], te.y_win, scoring='neg_brier_score', n_repeats=10, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=cols).sort_values(ascending=False); imp.to_csv(RES/'xgboost_permutation_importance.csv'); imp.round(5)

midpoint                0.00263
category_group          0.00022
slog_hours_to_event     0.00015
log_hours_since_open    0.00007
log_volume              0.00004
spread                  0.00001
log_open_interest       0.00001
has_prior_quote         0.00000
prior_candles_6h        0.00000
fresh_cross_f           0.00000
volume_missing          0.00000
log_volume_prev_6h     -0.00001
mid_change_6h_f        -0.00004
log_event_size         -0.00032
dtype: float64

## Descriptive findings: calibration, time to event, category, economics

In [8]:
def wilson(k, n, z=1.96):
    p = k / n; den = 1 + z*z/n; c = (p + z*z/(2*n)) / den; hw = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / den; return c - hw, c + hw
bins = [0.895, 0.92, 0.94, 0.96, 0.97, 0.98, 0.99, 1.0]
cal = d.assign(b=pd.cut(d.midpoint, bins)).groupby(['split', 'b'], observed=True).agg(n=('y_win','size'), wins=('y_win','sum'), mid=('midpoint','mean')).reset_index()
cal['win_rate'] = cal.wins / cal.n; cal[['lo','hi']] = [wilson(k, n) for k, n in zip(cal.wins, cal.n)]
cal.to_csv(RES/'calibration.csv', index=False)
hb = pd.cut(d.hours_to_event, [-1e9, 0, 1, 6, 24, 72, 1e9], labels=['event passed','0–1 h','1–6 h','6–24 h','1–3 days','> 3 days'])
tte = d.assign(hb=hb).groupby('hb', observed=True).agg(n=('y_win','size'), losses=('y_win', lambda s: int((1-s).sum())), mean_mid=('midpoint','mean')).reset_index()
tte['loss_rate'] = tte.losses / tte.n; tte['implied_loss'] = 1 - tte.mean_mid; tte.to_csv(RES/'loss_by_time_to_event.csv', index=False)
catt = d.groupby('category_group').agg(n=('y_win','size'), losses=('y_win', lambda s: int((1-s).sum())), mean_mid=('midpoint','mean'))
catt['loss_rate'] = catt.losses / catt.n; catt['implied_loss'] = 1 - catt.mean_mid; catt = catt.sort_values('n', ascending=False)
# event-level bootstrap CI for (observed − implied) loss rate per category: markets in one event (e.g. a ladder of strikes) move together
def cat_ci(g, B=1000):
    gev = g.groupby('event_slug').agg(l=('y_win', lambda s: (1-s).sum()), im=('midpoint', lambda s: (1-s).sum()), n=('y_win','size'))
    v = gev.values; out = []
    for _ in range(B):
        r = v[rng.integers(0, len(v), len(v))]; out.append((r[:,0].sum() - r[:,1].sum()) / r[:,2].sum())
    return pd.Series(dict(events=len(gev), gap_lo=np.percentile(out, 2.5), gap_hi=np.percentile(out, 97.5)))
catt = catt.join(d.groupby('category_group').apply(cat_ci)); catt['gap'] = catt.loss_rate - catt.implied_loss
catt.to_csv(RES/'loss_by_category.csv')
d['gross'] = d.y_win - d.ask
econ = d.groupby(['split','price_band']).agg(n=('gross','size'), wins=('y_win','sum'), gross_per_contract=('gross','mean'), total=('gross','sum')).reset_index()
econ.to_csv(RES/'economics_by_split_band.csv', index=False)
display(tte.round(4)); display(catt.round(4)); display(econ.round(4))

,hb,n,losses,mean_mid,loss_rate,implied_loss
0,event passed,364,10,0.9500,0.0275,0.0500
1,0–1 h,554,18,0.9461,0.0325,0.0539
2,1–6 h,2124,121,0.9426,0.0570,0.0574
3,6–24 h,3736,175,0.9503,0.0468,0.0497
4,1–3 days,3047,168,0.9426,0.0551,0.0574
5,> 3 days,526,21,0.9456,0.0399,0.0544


,n,losses,mean_mid,loss_rate,implied_loss,events,gap_lo,gap_hi,gap
category_group,,,,,,,,,
Sports,6674,382,0.9401,0.0572,0.0599,4040.0,-0.0099,0.0049,-0.0027
Economics,1518,7,0.9648,0.0046,0.0352,369.0,-0.0340,-0.0265,-0.0306
Financials,1066,78,0.9520,0.0732,0.0480,89.0,-0.0240,0.0842,0.0251
Commodities,381,19,0.9468,0.0499,0.0532,39.0,-0.0482,0.0537,-0.0033
Climate and Weather,268,12,0.9533,0.0448,0.0467,213.0,-0.0249,0.0244,-0.0019
Other,209,10,0.9439,0.0478,0.0561,77.0,-0.0383,0.0368,-0.0083
Crypto,148,2,0.9551,0.0135,0.0449,40.0,-0.0480,-0.0090,-0.0314
Entertainment,41,2,0.9589,0.0488,0.0411,17.0,-0.0464,0.0834,0.0077
Politics,33,1,0.9506,0.0303,0.0494,27.0,-0.0578,0.0545,-0.0191


,split,price_band,n,wins,gross_per_contract,total
0,test,90–94¢,699,638,-0.0146,-10.190
1,test,95–97¢,784,746,-0.0066,-5.200
2,test,98–99¢,816,801,-0.0052,-4.220
3,train,90–94¢,1516,1406,0.0007,1.070
4,train,95–97¢,1431,1380,0.0057,8.095
5,train,98–99¢,1411,1393,0.0012,1.720
6,validation,90–94¢,1079,982,-0.0171,-18.460
7,validation,95–97¢,1079,982,-0.0507,-54.680
8,validation,98–99¢,1536,1510,-0.0042,-6.480


## Does a model-based filter change the economics? (illustration only)
Buy only when the chosen model's win probability exceeds the ask by a margin picked on validation. One contract per
entry, bought at the displayed ask, held to settlement, before fees. Not an executable backtest.

In [9]:
best = scores[(scores.split == 'validation') & scores.model.str.contains('__')].sort_values('brier').iloc[0].model   # best fitted feature model
vb = val_preds[best]; trials = []
for mg in [0, 0.005, 0.01, 0.02, 0.03]:
    s = vb - va.ask.values > mg
    trials.append(dict(margin=mg, val_entries=int(s.sum()), val_gross_per=float((va.y_win.values - va.ask.values)[s].mean()) if s.sum() else np.nan))
trials = pd.DataFrame(trials); ok = trials[trials.val_entries >= 30]
mg = float(ok.sort_values('val_gross_per', ascending=False).iloc[0].margin) if len(ok) else 0.0
s = preds[best] - te.ask.values > mg; g = te.y_win.values - te.ask.values
strategy = pd.DataFrame([dict(rule='buy every eligible test entry', entries=len(te), losses=int((1-te.y_win).sum()), total_gross=g.sum(), per_contract=g.mean()),
                         dict(rule=f'{best}: p − ask > {mg}', entries=int(s.sum()), losses=int((1-te.y_win.values)[s].sum()), total_gross=g[s].sum(), per_contract=g[s].mean() if s.sum() else np.nan)])
strategy.to_csv(RES/'strategy_illustration.csv', index=False); trials.to_csv(RES/'strategy_validation_trials.csv', index=False); strategy.round(4)

,rule,entries,losses,total_gross,per_contract
0,buy every eligible test entry,2299,114,-19.61,-0.0085
1,elastic_net_logistic__with_volume: p − ask > 0.03,0,0,0.00,NaN


## Figures for the slides
Styled with `style_guide/ucla_style.py` (UCLA colors, Arial). Slide figures carry no title: the slide title states the
finding. Blue = the data, gray = the benchmark, gold = worse than the benchmark.

In [10]:
import sys; sys.path.insert(0, str(ROOT/'style_guide'))
import ucla_style as us
us.apply()

# 1 · calibration on test
fig, ax = us.figure((5.4, 4.3))
ct = cal[cal.split == 'test']
ax.plot([0.9, 1], [0.9, 1], color=us.REFERENCE, lw=1.4, ls='--', zorder=1)
ax.errorbar(ct.mid, ct.win_rate, yerr=[ct.win_rate - ct.lo, ct.hi - ct.win_rate], fmt='none', ecolor=us.DARK, lw=1.3, zorder=2)
ax.scatter(ct.mid, ct.win_rate, color=us.PRIMARY, s=60, zorder=3)
for k, (_, r) in enumerate(ct.iterrows()): us.label(ax, r.mid, 0.8625 + 0.0075 * (k % 2), f'n={r.n:,}', va='bottom', fontsize=9, color=us.GRAY_60)
ax.set(xlabel='Quoted midpoint (market probability)  ·  dashed: win rate = price', ylabel='Share of favorites that won', xlim=(0.9, 1.0), ylim=(0.86, 1.005))
ax.grid(axis='x', color=us.GRAY_10)
us.save(fig, FIG/'calibration_test.png')

# 2 · loss rate by time to the scheduled event
fig, ax = us.figure((5.8, 4.0)); x = np.arange(len(tte))
ax.bar(x - 0.19, tte.loss_rate * 100, 0.36, color=us.PRIMARY, label='Observed loss rate')
ax.bar(x + 0.19, tte.implied_loss * 100, 0.36, color=us.REFERENCE, label='Implied by price')
for i, r in tte.iterrows(): us.label(ax, i - 0.19, r.loss_rate * 100 + 0.12, f'{r.loss_rate*100:.1f}%', fontsize=9.5)
ax.set_xticks(x, [str(h).replace('event passed', 'started') for h in tte.hb]); ax.set_ylim(0, 7.2)
ax.set_ylabel('Favorites that lost (%)'); ax.set_xlabel('Time from entry to the scheduled event')
ax.legend(loc='upper left', ncol=2)
us.save(fig, FIG/'loss_by_time_to_event.png')

# 3 · every model vs the price (test Brier difference, event bootstrap 95% CI)
lab = {'price_only_logistic': 'Price-only logistic', 'training_win_rate': 'Training win rate',
       'elastic_net_logistic__no_volume': 'Elastic net', 'elastic_net_logistic__with_volume': 'Elastic net + volume',
       'random_forest__no_volume': 'Random forest', 'random_forest__with_volume': 'Random forest + volume',
       'xgboost__no_volume': 'XGBoost', 'xgboost__with_volume': 'XGBoost + volume'}
cm = cmp[cmp.model.isin(lab)].assign(label=lambda t: t.model.map(lab)).sort_values('brier_minus_midpoint', ascending=False)
fig, ax = us.figure((6.0, 4.1)); yy = np.arange(len(cm))
col = [us.PRIMARY if h < 0 else (us.ATTENTION if l > 0 else us.REFERENCE) for l, h in zip(cm.ci_low, cm.ci_high)]
ax.errorbar(cm.brier_minus_midpoint * 1e3, yy, xerr=[(cm.brier_minus_midpoint - cm.ci_low) * 1e3, (cm.ci_high - cm.brier_minus_midpoint) * 1e3],
            fmt='none', ecolor=us.DARK, lw=1.2)
ax.scatter(cm.brier_minus_midpoint * 1e3, yy, c=col, s=70, zorder=3, edgecolors=us.DARK, linewidths=0.6)
ax.axvline(0, color=us.GRAY_80, lw=1.2)
ax.set_yticks(yy, cm.label); ax.grid(axis='y', visible=False); ax.grid(axis='x', color=us.GRAY_10)
ax.set_xlabel('Test Brier score minus the quoted price (×1,000)')
ax.text(-0.04, len(cm) - 0.35, '← better than price', ha='right', fontsize=9.5, color=us.GRAY_60)
ax.text(0.04, len(cm) - 0.35, 'worse than price →', ha='left', fontsize=9.5, color=us.GRAY_60)
ax.set_ylim(-0.6, len(cm) - 0.1)
us.save(fig, FIG/'model_vs_price.png')

# 4 · loss rate by category (bars) vs price-implied rate (ticks)
cc = catt[catt.n >= 50].sort_values('loss_rate')
fig, ax = us.figure((6.2, 4.0)); yy = np.arange(len(cc))
bar_col = [us.ATTENTION if g > 0 and lo > 0 else us.PRIMARY for g, lo in zip(cc.gap, cc.gap_lo)]
ax.barh(yy, cc.loss_rate * 100, color=bar_col, height=0.6)
ax.scatter(cc.implied_loss * 100, yy, color=us.DARK, marker='|', s=320, lw=2.4, zorder=3)
xr = max(cc.loss_rate.max(), cc.implied_loss.max()) * 100 + 0.35
for i, (_, r) in enumerate(cc.iterrows()):
    ax.text(xr, i, f'{r.loss_rate*100:.1f}% vs {r.implied_loss*100:.1f}%  (n={int(r.n):,})', va='center', fontsize=9.5, color=us.GRAY_80)
ax.set_xlim(0, xr + 3.6); ax.set_yticks(yy, cc.index); ax.grid(axis='y', visible=False); ax.grid(axis='x', color=us.GRAY_10)
ax.set_xlabel('Favorites that lost (%)   ·   tick = rate implied by price')
us.save(fig, FIG/'loss_by_category.png')

# 5 · gross result per contract by period
es = d.groupby('split').gross.mean().reindex(['train', 'validation', 'test']) * 100
fig, ax = us.figure((5.4, 3.9))
ax.bar(['Train', 'Validation', 'Test'], es.values, color=[us.PRIMARY if v >= 0 else us.ATTENTION for v in es.values], width=0.55)
for i, v in enumerate(es.values): us.label(ax, i, v + (0.06 if v >= 0 else -0.06), f'{v:+.2f}¢'.replace('-', '−'), va='bottom' if v >= 0 else 'top', fontsize=11)
ax.axhline(0, color=us.GRAY_80, lw=1.2); ax.set_ylim(es.min() - 0.5, max(es.max(), 0) + 0.45)
ax.set_ylabel('Average gross result per contract (¢)')
us.save(fig, FIG/'economics_by_split.png')

# 6 · data coverage: hours with candles per day
cov = pd.read_csv(RES/'coverage_by_day.csv')
fig, ax = us.figure((5.8, 3.6))
ax.bar(cov.day.str[3:].astype(int).astype(str), cov.hours, color=us.PRIMARY, width=0.7)
for i, h in enumerate(cov.hours):
    if h == 0: us.label(ax, i, 0.3, 'none', fontsize=8.5, color=us.GRAY_60, rotation=90)
ax.set_ylim(0, 24.5); ax.set_yticks([0, 6, 12, 18, 24]); ax.set_ylabel('Hours with candles'); ax.set_xlabel('Day in October 2026 (UTC)')
us.save(fig, FIG/'coverage_by_day.png')
print(sorted(p.name for p in FIG.glob('*.png')))

['calibration_test.png', 'coverage_by_day.png', 'economics_by_split.png', 'loss_by_category.png', 'loss_by_time_to_event.png', 'model_vs_price.png']


In [11]:
summary = dict(chosen={f'{a}__{b}': v for (a, b), v in chosen.items()}, best_on_validation=best, strategy_margin=mg,
               n=d.groupby('split').y_win.agg(['size', lambda s: int((1-s).sum())]).set_axis(['markets','losses'], axis=1).to_dict('index'),
               collection_lag_median_h=float(d.collection_lag_h.median()), observed_after_close=float(d.observed_after_close.mean()))
(RES/'summary.json').write_text(json.dumps(summary, indent=2, default=str)); summary

{'chosen': {'elastic_net_logistic__no_volume': {'C': 0.01, 'l1': 0.5},
  'elastic_net_logistic__with_volume': {'C': 0.01, 'l1': 0.5},
  'random_forest__no_volume': {'max_depth': 4, 'min_samples_leaf': 20},
  'random_forest__with_volume': {'max_depth': 4, 'min_samples_leaf': 50},
  'xgboost__no_volume': {'max_depth': 2,
   'learning_rate': 0.03,
   'n_estimators': 200},
  'xgboost__with_volume': {'max_depth': 2,
   'learning_rate': 0.03,
   'n_estimators': 200}},
 'best_on_validation': 'elastic_net_logistic__with_volume',
 'strategy_margin': 0.03,
 'n': {'test': {'markets': 2299, 'losses': 114},
  'train': {'markets': 4358, 'losses': 179},
  'validation': {'markets': 3694, 'losses': 220}},
 'collection_lag_median_h': 2.475170281388889,
 'observed_after_close': 0.4951212443242199}